In [ ]:
"""
Step 3 
notebooks/3. feature_matrix_and_ch3_outputs_v2.ipynb

功能：
1) 特征矩阵构建与标准化（含自回归特征 sigma_t）
2) 第 3 章图表导出（3_6 ~ 3_9）
3) 第 3 章统计表导出（3_8 ~ 3_10）
4) 数据切分统计（表 3-11）
5) ADF 平稳性检验

输入: 02_finbert_features_added.csv
输出: 04_final_feature_matrix_v2.csv
"""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

# ==========================================
# 0. 环境与目录
# ==========================================
ROOT = Path.cwd()
PROJECT_DIR = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 当前使用的计算设备: {device}")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==========================================
# 1. 读取数据
# ==========================================
input_path = DATA_DIR / "02_finbert_features_added.csv"
print(f"📂 正在加载数据: {input_path}")
df = pd.read_csv(input_path)
print(f"   原始维度: {df.shape}")

# ==========================================
# 2. 特征衍生
# ==========================================
df["llm_composite_score"] = df["sentiment_class"] * (df["action_score"] / 10.0)
# target_sigma_t_plus_1[t] = sigma_(t+1)；shift(1) 得到 sigma_t
# 作为自回归特征加入基础组
# 注：首行会因 shift 产生 NaN
#     后续会删除
#
df["sigma_t"] = df["target_sigma_t_plus_1"].shift(1)
df = df.dropna(subset=["sigma_t"]).reset_index(drop=True)
print(f"✅ 加入 sigma_t 后维度: {df.shape} (首行 NaN 已删除)")

# ==========================================
# 3. 特征分组定义（sigma_t 纳入基础组）
# ==========================================
G1_cols = [
    "open",
    "close",
    "high",
    "low",
    "volume",
    "blocks-size",
    "avg-block-size",
    "n-transactions-total",
    "n-transactions-per-block",
    "hash-rate",
    "difficulty",
    "miners-revenue",
    "transaction-fees-usd",
    "n-unique-addresses",
    "n-transactions",
    "estimated-transaction-volume-usd",
    "total-bitcoins",
    "market-cap",
    "fng_value",
    "cbbi_value",
    "sigma_t",
]
G2_cols = G1_cols + ["finbert_sentiment_score"]
G3_cols = G1_cols + ["llm_composite_score"]
G3_M_cols = G1_cols + ["sentiment_class", "action_class", "action_score"]

all_feature_cols = list(set(G1_cols + G2_cols + G3_cols + G3_M_cols))
meta_cols = ["timestamp", "target_sigma_t_plus_1"]

print(
    f"📊 特征组维度: G1={len(G1_cols)}, G2={len(G2_cols)}, "
    f"G3={len(G3_cols)}, G3-M={len(G3_M_cols)}"
)

# ==========================================
# 4. 数据清理
# ==========================================
final_columns = meta_cols + all_feature_cols
df = df[final_columns].copy()
print(f"✅ 数据清理完成，当前 DataFrame 维度: {df.shape}")

# ==========================================
# 5. 时序划分与标准化（7:1:2）
# ==========================================
n_samples = len(df)
train_end = int(n_samples * 0.7)
val_end = int(n_samples * 0.8)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"📊 数据集划分 (7:1:2): Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
print(f"   训练期: {train_df['timestamp'].iloc[0]} ~ {train_df['timestamp'].iloc[-1]}")
print(f"   测试期: {test_df['timestamp'].iloc[0]} ~ {test_df['timestamp'].iloc[-1]}")

scaler = StandardScaler()
scaler.fit(train_df[all_feature_cols])

df_scaled = df.copy()
df_scaled[all_feature_cols] = scaler.transform(df[all_feature_cols])
print("✅ 特征标准化完成 (仅基于训练集拟合)")

# ==========================================
# 6. 保存
# ==========================================
scaler_path = MODEL_DIR / "feature_scaler_v2.pkl"
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler 模型已保存至: {scaler_path}")

ordered_cols = meta_cols + [col for col in final_columns if col not in meta_cols]
df_scaled = df_scaled[ordered_cols]

output_csv_path = DATA_DIR / "04_final_feature_matrix_v2.csv"
df_scaled.to_csv(output_csv_path, index=False)
print(f"💾 最终特征矩阵已成功导出至: {output_csv_path}")

# ==========================================
# 7. 验证：sigma_t 与目标相关性（原始量纲）
# ==========================================
df_raw = pd.read_csv(input_path)
df_raw["sigma_t"] = df_raw["target_sigma_t_plus_1"].shift(1)
df_raw = df_raw.dropna(subset=["sigma_t"])
corr_raw = df_raw["sigma_t"].corr(df_raw["target_sigma_t_plus_1"])
print("=" * 50)
print(f"验证: sigma_t 与 target 原始相关系数 = {corr_raw:.4f}")
print("=" * 50)
print("🎉 特征矩阵 v2 构建完成！")

✅ 当前使用的计算设备: cuda
📂 正在加载数据: D:\Workspace\03_Archive\btc-volatility-thesis\data\02_finbert_features_added.csv
   原始维度: (2337, 37)
✅ 加入 sigma_t 后维度: (2336, 39) (首行 NaN 已删除)
📊 特征组维度: G1=21, G2=22, G3=22, G3-M=24
✅ 数据清理完成，当前 DataFrame 维度: (2336, 28)
📊 数据集划分 (7:1:2): Train=1635, Val=233, Test=468
   训练期: 2018-02-02 ~ 2022-07-28
   测试期: 2023-03-19 ~ 2024-06-28
✅ 特征标准化完成 (仅基于训练集拟合)
💾 Scaler 模型已保存至: D:\Workspace\03_Archive\btc-volatility-thesis\models\feature_scaler_v2.pkl
💾 最终特征矩阵已成功导出至: D:\Workspace\03_Archive\btc-volatility-thesis\data\04_final_feature_matrix_v2.csv


验证: sigma_t 与 target 原始相关系数 = 0.5564
🎉 特征矩阵 v2 构建完成！


In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

warnings.filterwarnings("ignore")

# ==========================================
# 全局作图环境与字体设置
# ==========================================
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12
# 中文正文用 SimHei；坐标轴上的 $...$ 数学公式走 mathtext，须单独指定字体集，否则 σ、下标等会乱码
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["mathtext.default"] = "regular"

ROOT = Path.cwd()
PROJECT_DIR = ROOT.parent if ROOT.name == "notebooks" else ROOT
RESULT_DIR = PROJECT_DIR / "results"
RESULT_DIR.mkdir(exist_ok=True)

data_path = PROJECT_DIR / "data" / "02_finbert_features_added.csv"
print(f"📂 正在加载数据以进行统计制图: {data_path}")
df = pd.read_csv(data_path)

# 与特征矩阵构建保持一致
df["llm_composite_score"] = df["sentiment_class"] * (df["action_score"] / 10.0)
# 在统计分析阶段也补上 sigma_t 供可选分析/扩展
df["sigma_t"] = df["target_sigma_t_plus_1"].shift(1)

# 缺失值处理
df.fillna(method="ffill", inplace=True)
df.fillna(method="bfill", inplace=True)

# ==========================================
# 1. 核心特征分析 (图 3-6, 3-7, 3-8)
# ==========================================

# 图 3-6: LLM 复合因子分布
plt.figure(figsize=(8, 6))
sns.histplot(
    df["llm_composite_score"], bins=30, kde=True,
    color="#2c7bb6", edgecolor="black", alpha=0.7
)
plt.xlabel("LLM复合情感因子 ($S_{LLM}$)", fontsize=14)
plt.ylabel("频数 (Frequency)", fontsize=14)
plt.tight_layout()
plt.savefig(RESULT_DIR / "3_6_llm_composite_hist.png", dpi=300, format="png")
plt.close()
print("✅ 图 3-6 已保存。")

# 图 3-7: FinBERT vs LLM 分布 KDE
plt.figure(figsize=(8, 6))
sns.kdeplot(
    df["finbert_sentiment_score"], label="FinBERT ($S_{FB}$)",
    fill=True, color="#d7191c", alpha=0.4, linewidth=2
)
sns.kdeplot(
    df["llm_composite_score"], label="LLM复合因子 ($S_{LLM}$)",
    fill=True, color="#2c7bb6", alpha=0.4, linewidth=2
)
plt.xlabel("情感得分 (Sentiment Score)", fontsize=14)
plt.ylabel("概率密度 (Density)", fontsize=14)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.savefig(RESULT_DIR / "3_7_finbert_vs_llm_kde.png", dpi=300, format="png")
plt.close()
print("✅ 图 3-7 已保存。")

# 图 3-8: 核心特征相关性热力图（含滞后一期对数 Parkinson 波动率 σ_t）
core_cols_for_heatmap = [
    "target_sigma_t_plus_1",
    "sigma_t",
    "finbert_sentiment_score",
    "llm_composite_score",
    "close",
    "volume",
    "transaction-fees-usd",
    "hash-rate",
    "fng_value",
]
heatmap_labels = [
    r"$\sigma_{t+1}$",
    r"$\sigma_{t}$",
    r"$S_{FB}$",
    r"$S_{LLM}$",
    "Close Price",
    "Volume",
    "Tx Fees",
    "Hash Rate",
    "FNG Index",
]

corr_matrix = df[core_cols_for_heatmap].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdYlBu_r",
    xticklabels=heatmap_labels,
    yticklabels=heatmap_labels,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_DIR / "3_8_feature_correlation_heatmap.png", dpi=300, format="png")
plt.close()
print("✅ 图 3-8 已保存。")

# ==========================================
# 2. 时序领先性分析 (图 3-9)
# ==========================================
max_lags = 7
lags = np.arange(1, max_lags + 1)
corr_llm = [
    df["llm_composite_score"].corr(df["target_sigma_t_plus_1"].shift(-(k - 1)))
    for k in lags
]
corr_fb = [
    df["finbert_sentiment_score"].corr(df["target_sigma_t_plus_1"].shift(-(k - 1)))
    for k in lags
]

plt.figure(figsize=(10, 6))
bar_width = 0.35
x_indexes = np.arange(len(lags))

plt.bar(
    x_indexes - bar_width / 2,
    corr_fb,
    width=bar_width,
    label="FinBERT ($S_{FB}$)",
    color="#d7191c",
    alpha=0.8,
)
plt.bar(
    x_indexes + bar_width / 2,
    corr_llm,
    width=bar_width,
    label="LLM复合因子 ($S_{LLM}$)",
    color="#2c7bb6",
    alpha=0.8,
)

plt.xlabel("滞后天数 ($k$ days)", fontsize=14)
plt.ylabel("与对数极值波动率 ($\\sigma_{t+k}$) 的相关系数", fontsize=14)
plt.xticks(x_indexes, [f"Lag {k}" for k in lags])
plt.axhline(0, color="black", linewidth=1, linestyle="--")
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.savefig(RESULT_DIR / "3_9_lagged_correlation.png", dpi=300, format="png")
plt.close()
print("✅ 图 3-9 已保存。")

# ==========================================
# 3. 统计表 (3-8, 3-9, 3-10)
# ==========================================
groups_data = {
    "实验组": ["G1 (基础对照组)", "G2 (传统NLP情感组)", "G3 (LLM复合情感组)", "G3-M (LLM多维情感组)"],
    "包含特征": [
        "21项基础市场与链上特征(含sigma_t)",
        "G1特征 + FinBERT日度情感指数 ($S_{FB}$)",
        "G1特征 + LLM复合情感因子 ($S_{LLM}$)",
        "G1特征 + sentiment_class, action_class, action_score",
    ],
    "特征维度": [21, 22, 22, 24],
}
df_groups = pd.DataFrame(groups_data)
df_groups.to_csv(RESULT_DIR / "3_8_feature_groups.csv", index=False, encoding="utf-8-sig")
print("✅ 表 3-8 已导出。")

G1_cols = [
    "open", "close", "high", "low", "volume", "blocks-size", "avg-block-size",
    "n-transactions-total", "n-transactions-per-block", "hash-rate", "difficulty",
    "miners-revenue", "transaction-fees-usd", "n-unique-addresses", "n-transactions",
    "estimated-transaction-volume-usd", "total-bitcoins", "market-cap", "fng_value", "cbbi_value", "sigma_t"
]
all_used_cols = G1_cols + ["finbert_sentiment_score", "llm_composite_score", "target_sigma_t_plus_1"]

# 描述性统计在原始量纲上做，便于论文解释
df_desc = df[all_used_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
df_desc.columns = ["均值 (Mean)", "标准差 (Std)", "最小值 (Min)", "25%分位数", "中位数 (Median)", "75%分位数", "最大值 (Max)"]
df_desc.index.name = "特征变量 (Feature)"
df_desc.to_csv(RESULT_DIR / "3_9_descriptive_stats.csv", encoding="utf-8-sig")
print("✅ 表 3-9 已导出。")

# Granger 因果检验
df_granger = pd.DataFrame({
    "current_sigma": df["target_sigma_t_plus_1"].shift(1),
    "S_LLM": df["llm_composite_score"],
    "S_FB": df["finbert_sentiment_score"],
}).dropna()

maxlag = 3
results_list = []
for factor in ["S_LLM", "S_FB"]:
    test_res = grangercausalitytests(df_granger[["current_sigma", factor]], maxlag=maxlag, verbose=False)
    for lag in range(1, maxlag + 1):
        f_val = test_res[lag][0]["ssr_ftest"][0]
        p_val = test_res[lag][0]["ssr_ftest"][1]
        results_list.append({
            "检验关系": f"{factor} -> $\\sigma$",
            "滞后阶数 (Lag)": lag,
            "F-Statistic": round(f_val, 4),
            "P-Value": round(p_val, 4),
            "显著性结论": "显著 (H1)" if p_val < 0.05 else "不显著 (H0)",
        })

pd.DataFrame(results_list).to_csv(RESULT_DIR / "3_10_granger_causality.csv", index=False, encoding="utf-8-sig")
print("✅ 表 3-10 已导出。")

# ==========================================
# 4. ADF 平稳性检验（原 notebook 3 的职能）
# ==========================================
adf_cols = ["target_sigma_t_plus_1", "sigma_t", "finbert_sentiment_score", "llm_composite_score"]
df_adf = df[adf_cols].dropna()

print(f"\n✅ ADF 检验有效样本量: {len(df_adf)}")
print("| 特征名称 | ADF 统计量 | p-value | 1% 临界值 | 5% 临界值 | 结论(是否平稳) |")
print("| :--- | :--- | :--- | :--- | :--- | :--- |")
for col in adf_cols:
    res = adfuller(df_adf[col], autolag="AIC")
    adf_stat, p_value = res[0], res[1]
    crit_1, crit_5 = res[4]["1%"], res[4]["5%"]
    st = "平稳" if p_value < 0.05 else "不平稳"
    print(f"| `{col}` | {adf_stat:.4f} | {p_value:.4e} | {crit_1:.4f} | {crit_5:.4f} | **{st}** |")

print("\n🎉 图表、统计分析与 ADF 检验执行完毕。")

📂 正在加载数据以进行统计制图: D:\Workspace\03_Archive\btc-volatility-thesis\data\02_finbert_features_added.csv


✅ 图 3-6 已保存。


✅ 图 3-7 已保存。


✅ 图 3-8 已保存。


✅ 图 3-9 已保存。
✅ 表 3-8 已导出。
✅ 表 3-9 已导出。
✅ 表 3-10 已导出。
🎉 图表与统计分析执行完毕。


In [3]:
# 已合并到 Cell 1；此单元保留为空占位

📊 毕业论文表 3-11: 数据集划分及其时间分布统计


,数据集类型,样本量,占比,开始日期,结束日期
0,训练集 (Train Set),1635,70%,2018-02-02,2022-07-28
1,验证集 (Validation Set),233,10%,2022-07-29,2023-03-18
2,测试集 (Test Set),468,20%,2023-03-19,2024-06-28


✅ 验证结果: 总样本数(2336) = 各部分之和(2336)


In [4]:
# 已合并到 Cell 1；此单元保留为空占位

📂 正在加载数据: D:\Workspace\03_Archive\btc-volatility-thesis\data\04_final_feature_matrix_v2.csv
✅ 缺失值处理完毕，参与 ADF 检验的有效样本量: 2336

| 特征名称 | ADF 统计量 | p-value | 1% 临界值 | 5% 临界值 | 结论(是否平稳) |
| :--- | :--- | :--- | :--- | :--- | :--- |


| `target_sigma_t_plus_1` | -5.6374 | 1.0554e-06 | -3.4332 | -2.8628 | **平稳** |


| `sigma_t` | -5.5983 | 1.2821e-06 | -3.4332 | -2.8628 | **平稳** |
| `finbert_sentiment_score` | -6.8459 | 1.7431e-09 | -3.4332 | -2.8628 | **平稳** |


| `llm_composite_score` | -5.1317 | 1.2099e-05 | -3.4332 | -2.8628 | **平稳** |
